# 03 散布図と相関係数：Excel の CORREL と同じ値になるか

**この課題で体験するデータ工学的な難しさ**
1. 相関係数は道具が変わっても同じ値になるはず。Excel の分析ツール（第 8 回）と照合して「再現できた」と言えるようにする。
2. 人口が多い県は何でも多い。「人口あたり」に直すと相関は消えるか、残るか。
3. 相関があっても因果ではない。散布図の外れ値（東京都）を除くとどうなるかを見る。

**やること**：上から順に実行。「★」の行だけ書き換えてよい。

## 1. 表を読む
人口・面積・病院数・医師数（すべて 2020 年、出典は `data/SOURCES.md`）。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("../data/BIZUDPGothic-Regular.ttf")   # 日本語フォント（同梱）
plt.rcParams["font.family"] = "BIZ UDPGothic"
df = pd.read_csv("../data/todofuken_iryo.csv", dtype={"都道府県コード": str})
df.head()

## 2. 2 つの列を選んで散布図
★ `x` と `y` を別の列名に書き換える。使える列：`人口_2020` `面積_km2` `病院数_2020` `医師数_2020`

In [ ]:
x = "人口_2020"     # ★
y = "医師数_2020"   # ★
plt.figure(figsize=(6, 5))
plt.scatter(df[x], df[y])
for _, r in df.iterrows():
    plt.annotate(r["都道府県"], (r[x], r[y]), fontsize=7)
plt.xlabel(x); plt.ylabel(y); plt.title(f"{x} と {y}")
plt.tight_layout(); plt.show()

## 3. 相関係数（Excel の `=CORREL(範囲1, 範囲2)` と同じ計算）
Excel で同じ 2 列の CORREL を取り、小数第 4 位まで一致するか確かめる。

In [ ]:
r = df[x].corr(df[y])
print(f"{x} と {y} の相関係数 r = {r:.4f}")

## 4. 全部の組み合わせを一度に見る（相関行列）

In [ ]:
df[["人口_2020", "面積_km2", "病院数_2020", "医師数_2020"]].corr().round(3)

## 5. 人口あたりに直すと相関は残るか
「多い県は何でも多い」を取り除く。

In [ ]:
df["病院数_10万人あたり"] = df["病院数_2020"] / df["人口_2020"] * 100000
df["医師数_10万人あたり"] = df["医師数_2020"] / df["人口_2020"] * 100000
print("そのまま      r =", round(df["病院数_2020"].corr(df["医師数_2020"]), 4))
print("10万人あたり  r =", round(df["病院数_10万人あたり"].corr(df["医師数_10万人あたり"]), 4))

## 6. 外れ値を除くと
★ 除く都道府県を書き換える。

In [ ]:
除く = ["東京都"]   # ★
sub = df[~df["都道府県"].isin(除く)]
print("除く前 r =", round(df[x].corr(df[y]), 4))
print("除いた後 r =", round(sub[x].corr(sub[y]), 4))

## 7. 確かめること（提出用に 3 行で書く）
- Excel の CORREL（または分析ツールの相関）と、ここで出た r は一致したか。一致しなければ、どの範囲を選んだかを見直す。
- 人口あたりに直すと、病院数と医師数の相関はどう変わったか。それは何を意味するか。
- 相関が強い組み合わせに「原因と結果」があると言えるか。言えないとしたら、何が両方を増やしているか。